# Azure AI Vision Services Lab

This notebook provides hands-on exercises for Azure AI Vision services using the **REST APIs and SDKs**.

## ⚠️ Deprecation Notices

| Service | Status | Retirement Date | Suggested Alternatives |
|---------|--------|----------------|------------------------|
| **OCR (Image Analysis 4.0)** | Deprecated | September 25, 2028 | [Document Intelligence](https://learn.microsoft.com/azure/ai-services/document-intelligence/prebuilt/read), [Content Understanding](https://learn.microsoft.com/azure/ai-services/content-understanding/overview) |
| **OCR (Computer Vision v1.0–v3.1)** | Deprecated | September 13, 2026 | [Document Intelligence](https://learn.microsoft.com/azure/ai-services/document-intelligence/prebuilt/read) |
| **Image Analysis 4.0** | Deprecated | September 25, 2028 | [GPT models in Microsoft Foundry](https://learn.microsoft.com/azure/ai-foundry/concepts/foundry-models-overview), [Content Understanding](https://learn.microsoft.com/azure/ai-services/content-understanding/overview) |
| **Azure Face API** | ✅ Active | No retirement date | N/A |
| **Azure Video Indexer** | ✅ Active | No retirement date | N/A |

## 1. Load Environment Variables and Python Libraries

This cell loads the configuration settings from your `.env` file, ensuring that your API credentials and file paths are available for subsequent code cells.

In [ ]:
# Install Azure Image Analysis Python Libraries
! pip install azure-ai-vision-imageanalysis==1.0.0 matplotlib==3.10.8 pillow==11.3.0

In [ ]:
# Load python libraries
import os
from pathlib import Path
from dotenv import load_dotenv
from PIL import Image
import matplotlib.pyplot as plt
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential

# Load environment variables
candidate_env_files = [Path.cwd() / ".env"]
candidate_env_files.extend(parent / ".env" for parent in Path.cwd().parents)
env_file = next((path for path in candidate_env_files if path.is_file()), None)

if env_file is None:
    raise FileNotFoundError(
        f"Could not find a .env file from notebook working directory {Path.cwd()}. Add the shared .env file to the lab workspace root and rerun this cell."
    )

load_dotenv(env_file, override=True)

COMPUTER_VISION_API_KEY = os.getenv("COMPUTER_VISION_API_KEY")
COMPUTER_VISION_ENDPOINT = os.getenv("COMPUTER_VISION_ENDPOINT")
VIDEO_INDEXER_ACCOUNT_ID = os.getenv("VIDEO_INDEXER_ACCOUNT_ID")
VIDEO_INDEXER_LOCATION = os.getenv("VIDEO_INDEXER_LOCATION")
VIDEO_INDEXER_TOKEN = os.getenv("VIDEO_INDEXER_TOKEN")

required_env_vars = {
    "COMPUTER_VISION_API_KEY": COMPUTER_VISION_API_KEY,
    "COMPUTER_VISION_ENDPOINT": COMPUTER_VISION_ENDPOINT,
}
missing_env_vars = [name for name, value in required_env_vars.items() if not value]

if missing_env_vars:
    raise EnvironmentError(
        "Missing required environment variables after loading .env: " + ", ".join(missing_env_vars)
    )

## 2. Extract Text from Images
Optical Character Recognition (OCR)

> ⚠️ **Deprecation Notice:** The Image Analysis 4.0 OCR API will be retired on **September 25, 2028**. Legacy Computer Vision API versions (v1.0–v3.1) will be retired on **September 13, 2026**. Consider migrating to [Azure AI Document Intelligence](https://learn.microsoft.com/azure/ai-services/document-intelligence/prebuilt/read) or [Azure Content Understanding](https://learn.microsoft.com/azure/ai-services/content-understanding/overview).

In [ ]:
# Set the path to the image you want to analyze
IMAGEPATH_OCR= "images/ocr_image.png"

In [ ]:
# Set the values of your computer vision endpoint and computer vision key as environment variables:
try:
    endpoint = COMPUTER_VISION_ENDPOINT
    key = COMPUTER_VISION_API_KEY
except KeyError:
    print("Missing environment variable 'VISION_ENDPOINT' or 'VISION_KEY'")
    print("Set them before running this sample.")
    exit()

# Create an Image Analysis client
client = ImageAnalysisClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

# Read the local image file and analyze it. This will be a synchronously (blocking) call.
with open(IMAGEPATH_OCR, "rb") as f:
    image_data = f.read()

result = client.analyze(
    image_data=image_data,
    visual_features=[VisualFeatures.CAPTION, VisualFeatures.READ],
    gender_neutral_caption=True,  # Optional (default is False)
)

print("Image analysis results:")
# Print caption results to the console
print(" Caption:")
if result.caption is not None:
    print(f"   '{result.caption.text}', Confidence {result.caption.confidence:.4f}")

# Print text (OCR) analysis results to the console
print(" Read:")
if result.read is not None:
    for line in result.read.blocks[0].lines:
        print(f"   Line: '{line.text}', Bounding box {line.bounding_polygon}")
        for word in line.words:
            print(f"     Word: '{word.text}', Bounding polygon {word.bounding_polygon}, Confidence {word.confidence:.4f}")

## 3. Extract Tags from Images
Extract common tags and objects detected in images

> ⚠️ **Deprecation Notice:** The Image Analysis 4.0 service will be retired on **September 25, 2028**. Consider migrating to [GPT models in Microsoft Foundry](https://learn.microsoft.com/azure/ai-foundry/concepts/foundry-models-overview) or [Azure Content Understanding](https://learn.microsoft.com/azure/ai-services/content-understanding/overview).

In [ ]:
# Set the path to the image you want to analyze
IMAGEPATH_TAG = "images/face-reco-mobile.jpg"

In [ ]:
# Create an Image Analysis client.
client = ImageAnalysisClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

# Read the local image file and analyze it.
with open(IMAGEPATH_TAG, "rb") as f:
    image_data = f.read()

# Do 'Tags' analysis on an image stream. This will be a synchronously (blocking) call.
result = client.analyze(
    image_data=image_data,
    visual_features=[VisualFeatures.TAGS],
    language="en",  # Optional. See https://aka.ms/cv-languages for supported languages.
)

# Print Tags analysis results to the console
print("Image analysis results:")
print(" Tags:")
if result.tags is not None:
    for tag in result.tags.list:
        print(f"   '{tag.name}', Confidence {tag.confidence:.4f}")        
print(f" Image height: {result.metadata.height}")
print(f" Image width: {result.metadata.width}")
print(f" Model version: {result.model_version}")

## 4. Add Dense Captions to Images
Provide Dense captions to images

> ⚠️ **Deprecation Notice:** The Image Analysis 4.0 service will be retired on **September 25, 2028**.

In [ ]:
# Set the path to the image you want to analyze
IMAGEPATH_DENSECAPS = "images/ms-redmond.jpg"

In [ ]:
# Create an Image Analysis client.
client = ImageAnalysisClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

# Load image to analyze into a 'bytes' object.
with open(IMAGEPATH_DENSECAPS, "rb") as f:
    image_data = f.read()

# Extract multiple captions, each for a different area of the image.
# This will be a synchronously (blocking) call.
result = client.analyze(
    image_data=image_data,
    visual_features=[VisualFeatures.DENSE_CAPTIONS],
    gender_neutral_caption=True,  # Optional (default is False)
)

# Print dense caption results to the console. The first caption always
# corresponds to the entire image. The rest correspond to sub regions.
print("Image analysis results:")
print(" Dense Captions:")
if result.dense_captions is not None:
    for caption in result.dense_captions.list:
        print(f"   '{caption.text}', {caption.bounding_box}, Confidence: {caption.confidence:.4f}")
        
print(f" Image height: {result.metadata.height}")
print(f" Image width: {result.metadata.width}")
print(f" Model version: {result.model_version}")

## 05. Add Captions to Images
Generate captions from an image using Azure AI Vision.

> ⚠️ **Deprecation Notice:** The Image Analysis 4.0 service will be retired on **September 25, 2028**.

In [ ]:
# Set the path to the image you want to analyze
IMAGEPATH_CAPTIONS = "images/ms-redmond.jpg"

In [ ]:
# Create an Image Analysis client
client = ImageAnalysisClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

# Load image to analyze into a 'bytes' object
with open(IMAGEPATH_CAPTIONS, "rb") as f:
    image_data = f.read()

# Get a caption for the image. This will be a synchronously (blocking) call.
result = client.analyze(
    image_data=image_data,
    visual_features=[VisualFeatures.CAPTION],
    gender_neutral_caption=True,  # Optional (default is False)
)

# Print caption results to the console
print("Image analysis results:")
print(" Caption:")
if result.caption is not None:
    print(f"   '{result.caption.text}', Confidence {result.caption.confidence:.4f}")

print(f" Image height: {result.metadata.height}")
print(f" Image width: {result.metadata.width}")
print(f" Model version: {result.model_version}")

## 06. Create Smart-Cropped Images

This sample demonstrates how to find representatives sub-regions of image files for thumbnail generation. It uses an asynchronous client.

> ⚠️ **Deprecation Notice:** The Image Analysis 4.0 service will be retired on **September 25, 2028**.

In [ ]:
# Set the path to the image you want to analyze
IMAGEPATH_CROP = "images/cropped-original.png"

In [ ]:
# Create an Image Analysis client
client = ImageAnalysisClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

# Load image to analyze into a 'bytes' object
with open(IMAGEPATH_CROP, "rb") as f:
    image_data = f.read()

# Do Smart Cropping analysis on an image stream. This will be a synchronously (blocking) call.
result = client.analyze(
    image_data=image_data,
    visual_features=[VisualFeatures.SMART_CROPS],
    smart_crops_aspect_ratios=[0.9, 1.33],  # Optional. Specify one more desired aspect ratios
)

# Print smart crop analysis results to the console
print("Image analysis results:")
print(" Smart Cropping:")
if result.smart_crops is not None:
    for smart_crop in result.smart_crops.list:
        print(f"   Aspect ratio {smart_crop.aspect_ratio}: Smart crop {smart_crop.bounding_box}")
        
print(f" Image height: {result.metadata.height}")
print(f" Image width: {result.metadata.width}")
print(f" Model version: {result.model_version}")

# Display the original and cropped image side by side
if result.smart_crops is not None and len(result.smart_crops.list) > 0:
    original_image = Image.open(IMAGEPATH_CROP)
    first_crop = result.smart_crops.list[0].bounding_box
    cropped_image = original_image.crop((
        first_crop.x,
        first_crop.y,
        first_crop.x + first_crop.width,
        first_crop.y + first_crop.height
    ))
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(original_image)
    axes[0].axis("off")
    axes[0].set_title("Original Image")
    axes[1].imshow(cropped_image)
    axes[1].axis("off")
    axes[1].set_title(f"Smart Crop (Aspect ratio: {result.smart_crops.list[0].aspect_ratio})")
    plt.tight_layout()
    plt.show()